# 12 – Riippuuko tulos tunnistettavimmista edustajista?

**Tehtävä:** tarkistaa, johtuuko erottuvuuden kasvu (muistio 05) muutamasta edustajasta, joiden kieli on poikkeuksellisen helppo tunnistaa. Jos näin olisi, kasvu katoaisi, kun nämä edustajat poistetaan.

**Syöte:**
- `Data/processed/party_oof.parquet`: puhekohtaiset testiennusteet, 8 toistoa per kausi (muistio 05)
- `Data/processed/counts.npz` ja `counts_index.parquet` (muistio 04)
- `results/party_summary.csv` (muistio 05)

**Tulos:** `results/tunnistettavat_uudelleenopetus.csv` (osa B). Osan A tulokset vain muistioon.

**Ajoaika:** osa A noin minuutti. Osa B opettaa mallit uudelleen (3 kautta × 12 toistoa × 5 mallia) ja vie noin 4 minuuttia; se luetaan tiedostosta, jos tiedosto on jo olemassa (`UUDELLEEN = True` laskee uudelleen).

**Tietosuoja:** edustajia ei nimetä eikä aseteta järjestykseen. Muistio tulostaa vain puoluetason määriä, eikä edustajien tunnisteita tallenneta.

## Menetelmä

1. **Tunnistettavuus:** edustajan kaikkien testipuheiden keskimääräinen todennäköisyys omalle puolueelle kahdeksan toiston yli. Mitä suurempi luku, sitä helpommin malli tunnistaa edustajan puolueen.
2. **Poisto:** kunkin puolueen tunnistettavimmat 10 % ja 20 % edustajista kaudella (vähintään yksi edustaja puoluetta kohden).
3. **Osa A, uudelleenlaskenta:** tarkkuus ja erottuvuus lasketaan muistion 05 testiennusteista ilman poistettuja edustajia. Vertailuksi poistetaan sama määrä satunnaisia edustajia.
4. **Osa B, uudelleenopetus:** tunnistettavimmat 10 % poistetaan koko aineistosta, ja mallit opetetaan uudelleen samalla tavalla kuin muistiossa 05 (8 toistoa oikeilla ja 4 sekoitetuilla nimikkeillä). Näin poistetut edustajat eivät vaikuta myöskään opetukseen.

Tunnistettavimpien poistaminen laskee erottuvuutta kaikilla kausilla, koska poistetuksi valitaan juuri ne, jotka malli tunnistaa parhaiten. Kiinnostava kysymys on, **säilyykö kausien välinen ero**.

In [1]:
import os
import sys
from itertools import combinations

import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline

sys.path.insert(0, "../src")
import config

KANSIO = "../Data/processed/"
TULOKSET = "../results/"
KAUDET = ["2015-2019", "2019-2023", "2023-2027"]
SIEMENET = {"2015-2019": 100, "2019-2023": 101, "2023-2027": 102}
PUOLUEET = list(config.MAIN_PARTIES)
SARAKKEET = ["p_" + p for p in PUOLUEET]

# Asetukset
OSUUDET = [0.10, 0.20]   # poistettava osuus kunkin puolueen edustajista
ARVONTOJA = 10           # satunnaisen poiston toistot (osa A)
UUDELLEEN = False        # True: laske osa B uudelleen, vaikka tulostiedosto on jo olemassa

# Muistion 05 asetukset (osa B)
MIN_FRAASEJA, N_OPETUS, OSIOITA, EDUSTAJA_OSUUS, C = 20, 300, 5, 0.8, 2.0
TOISTOT, SATUNNAISET = 8, 4

oof = pd.read_parquet(KANSIO + "party_oof.parquet")
indeksi = pd.read_parquet(KANSIO + "counts_index.parquet")
oof = oof.merge(indeksi[["speech_id", "person_id"]], on="speech_id", how="left")
sarake = oof["label"].map({p: i for i, p in enumerate(PUOLUEET)}).to_numpy()
oof["oma_p"] = oof[SARAKKEET].to_numpy()[np.arange(len(oof)), sarake]
print("Testiennusteita:", len(oof), " toistoja:", oof["rep"].nunique())

Testiennusteita: 839170  toistoja: 8


## Tunnistettavuus ja poistettavat edustajat

Taulukossa on poistettujen edustajien määrä ja heidän osuutensa kauden puheista. Viimeiset sarakkeet vertaavat poistettujen ja muiden keskimääräistä tunnistettavuutta.

In [2]:
tunnistettavuus = oof.groupby(["term", "label", "person_id"])["oma_p"].mean().reset_index()

POIS = {}
for (kausi, puolue), ryhma in tunnistettavuus.groupby(["term", "label"]):
    ryhma = ryhma.sort_values("oma_p", ascending=False)
    for osuus in OSUUDET:
        n = max(1, int(round(osuus * len(ryhma))))
        POIS.setdefault((kausi, osuus), set()).update(ryhma["person_id"].iloc[:n])

puheita = indeksi[indeksi["party"].isin(PUOLUEET)].groupby("term").size()
rivit = []
for (kausi, osuus), pois in POIS.items():
    t = tunnistettavuus[tunnistettavuus["term"] == kausi]
    poistettu = t["person_id"].isin(pois)
    rivit.append({
        "kausi": kausi, "osuus": f"{osuus:.0%}", "poistettuja": len(pois),
        "puheista_%": 100 * indeksi[(indeksi["term"] == kausi) & indeksi["person_id"].isin(pois)].shape[0] / puheita[kausi],
        "tunnistettavuus_poistetut": t.loc[poistettu, "oma_p"].mean(),
        "tunnistettavuus_muut": t.loc[~poistettu, "oma_p"].mean(),
    })
pd.DataFrame(rivit).sort_values(["osuus", "kausi"]).round(3).reset_index(drop=True)

,kausi,osuus,poistettuja,puheista_%,tunnistettavuus_poistetut,tunnistettavuus_muut
0,2015-2019,10%,22,13.610,0.187,0.151
1,2019-2023,10%,21,8.823,0.215,0.163
2,2023-2027,10%,23,12.697,0.216,0.168
3,2015-2019,20%,41,21.572,0.180,0.149
4,2019-2023,20%,42,18.252,0.205,0.159
5,2023-2027,20%,42,22.724,0.209,0.164


## Osa A: uudelleenlaskenta ilman poistettuja edustajia

Erottuvuus lasketaan samalla tavalla kuin muistiossa 05: puolueparin erottuvuus on 2·AUC − 1, ja kauden luku on 28 puolueparin keskiarvo. Satunnaisessa poistossa poistetaan kustakin puolueesta yhtä monta edustajaa kuin 10 %:n poistossa, ja tulos on kymmenen arvonnan keskiarvo.

In [3]:
def parien_erottuvuus(oikeat, todnak, luokat):
    """Erottuvuus d = 2*AUC - 1 jokaiselle luokkaparille (sama kuin muistiossa 05)."""
    log_p = np.log(np.clip(todnak, 1e-12, 1))
    tulos = {}
    for i, j in combinations(range(len(luokat)), 2):
        rivit = (oikeat == luokat[i]) | (oikeat == luokat[j])
        if rivit.sum() == 0 or len(np.unique(oikeat[rivit])) < 2:
            continue
        pisteet = log_p[rivit, i] - log_p[rivit, j]
        tulos[(luokat[i], luokat[j])] = 2 * roc_auc_score(oikeat[rivit] == luokat[i], pisteet) - 1
    return tulos


def mittarit(oikeat, todnak):
    arvaus = np.array(PUOLUEET)[todnak.argmax(axis=1)]
    return {"accuracy": float((arvaus == oikeat).mean()),
            "polarisation": float(np.mean(list(parien_erottuvuus(oikeat, todnak, PUOLUEET).values())))}


def mittarit_ilman(osa, pois):
    jaljella = osa[~osa["person_id"].isin(pois)]
    return mittarit(jaljella["label"].to_numpy(), jaljella[SARAKKEET].to_numpy())


poistomaara = {}
for (kausi, puolue), ryhma in tunnistettavuus.groupby(["term", "label"]):
    poistomaara[(kausi, puolue)] = max(1, int(round(OSUUDET[0] * len(ryhma))))

rivit = []
for (kausi, toisto), osa in oof.groupby(["term", "rep"]):
    rivit.append({"term": kausi, "rep": toisto, "versio": "kaikki", **mittarit_ilman(osa, set())})
    for osuus in OSUUDET:
        rivit.append({"term": kausi, "rep": toisto, "versio": f"ilman tunnistettavinta {osuus:.0%}",
                      **mittarit_ilman(osa, POIS[(kausi, osuus)])})
    rng = np.random.default_rng(SIEMENET[kausi] * 100 + toisto)
    arvonnat = []
    for _ in range(ARVONTOJA):
        pois = set()
        for puolue in PUOLUEET:
            edustajat = tunnistettavuus.loc[(tunnistettavuus["term"] == kausi) & (tunnistettavuus["label"] == puolue), "person_id"].to_numpy()
            pois.update(rng.choice(edustajat, size=poistomaara[(kausi, puolue)], replace=False))
        arvonnat.append(mittarit_ilman(osa, pois))
    rivit.append({"term": kausi, "rep": toisto, "versio": f"ilman satunnaista {OSUUDET[0]:.0%}",
                  **pd.DataFrame(arvonnat).mean().to_dict()})
osa_a = pd.DataFrame(rivit)

In [4]:
def yhteenveto(taulu, mittari):
    """Keskiarvo ja 10.–90. persentiili toistojen yli sekä muutos kaudesta 2015–19 kauteen 2023–27."""
    g = taulu.groupby(["versio", "term"])[mittari]
    t = pd.DataFrame({"ka": g.mean(), "p10": g.quantile(0.1), "p90": g.quantile(0.9)}).round(3)
    t = t.apply(lambda r: f"{r['ka']:.3f} ({r['p10']:.3f}–{r['p90']:.3f})", axis=1).unstack("term")[KAUDET]
    keskiarvot = g.mean().unstack("term")
    t["muutos 2015–19 → 2023–27"] = (keskiarvot["2023-2027"] - keskiarvot["2015-2019"]).round(3)
    return t


JARJESTYS = ["kaikki", "ilman satunnaista 10%", "ilman tunnistettavinta 10%", "ilman tunnistettavinta 20%"]
print("Erottuvuus, keskiarvo (10.–90. persentiili)")
yhteenveto(osa_a, "polarisation").reindex(JARJESTYS)

Erottuvuus, keskiarvo (10.–90. persentiili)


term,2015-2019,2019-2023,2023-2027,muutos 2015–19 → 2023–27
versio,,,,
kaikki,0.391 (0.369–0.408),0.451 (0.402–0.485),0.558 (0.527–0.595),0.167
ilman satunnaista 10%,0.392 (0.367–0.413),0.451 (0.402–0.488),0.560 (0.532–0.596),0.167
ilman tunnistettavinta 10%,0.345 (0.328–0.360),0.414 (0.372–0.442),0.508 (0.473–0.547),0.163
ilman tunnistettavinta 20%,0.324 (0.309–0.338),0.387 (0.348–0.416),0.483 (0.446–0.528),0.158


In [5]:
print("Tarkkuus, keskiarvo (10.–90. persentiili)")
yhteenveto(osa_a, "accuracy").reindex(JARJESTYS)

Tarkkuus, keskiarvo (10.–90. persentiili)


term,2015-2019,2019-2023,2023-2027,muutos 2015–19 → 2023–27
versio,,,,
kaikki,0.259 (0.253–0.268),0.301 (0.291–0.311),0.324 (0.308–0.336),0.064
ilman satunnaista 10%,0.260 (0.253–0.270),0.302 (0.290–0.314),0.325 (0.310–0.336),0.065
ilman tunnistettavinta 10%,0.238 (0.227–0.246),0.290 (0.279–0.299),0.301 (0.285–0.314),0.062
ilman tunnistettavinta 20%,0.229 (0.216–0.236),0.279 (0.271–0.289),0.285 (0.269–0.298),0.056


## Osa B: mallit opetetaan uudelleen ilman tunnistettavimpia 10 %

Funktiot ovat samat kuin muistiossa 05; ainoa ero on, että poistetut edustajat eivät ole aineistossa lainkaan. Satunnaislukujen siemenet ovat samat kuin muistiossa 05, mutta koska aineisto on eri, toistot eivät ole parittaisia alkuperäisten kanssa.

In [6]:
def tee_malli():
    return make_pipeline(TfidfTransformer(sublinear_tf=True), LogisticRegression(C=C, max_iter=3000))


def yksi_toisto(X, puheet, rng, sekoita):
    """Kuten muistiossa 05: 80 % edustajista, ristiinvalidointi edustajittain, tasapainotettu opetus."""
    if sekoita:
        puheet = puheet.copy()
        edustajan_puolue = puheet.groupby("person_id")["label"].agg(lambda x: x.mode().iat[0])
        sekoitettu = pd.Series(rng.permutation(edustajan_puolue.values), index=edustajan_puolue.index)
        puheet["label"] = puheet["person_id"].map(sekoitettu).values
    valitut = set()
    for luokka in PUOLUEET:
        edustajat = puheet.loc[puheet["label"] == luokka, "person_id"].unique()
        maara = max(2, int(round(EDUSTAJA_OSUUS * len(edustajat))))
        valitut.update(rng.choice(edustajat, size=min(maara, len(edustajat)), replace=False))
    rivit = puheet["person_id"].isin(valitut).values
    puheet, X = puheet[rivit], X[rivit]
    oikeat, ryhmat = puheet["label"].values, puheet["person_id"].values
    todnak = np.zeros((len(puheet), len(PUOLUEET)))
    jako = StratifiedGroupKFold(n_splits=OSIOITA, shuffle=True, random_state=int(rng.integers(1e9)))
    for opetus, testi in jako.split(X, oikeat, ryhmat):
        opetusrivit = []
        for luokka in PUOLUEET:
            omat = opetus[oikeat[opetus] == luokka]
            opetusrivit.extend(rng.choice(omat, size=min(N_OPETUS, len(omat)), replace=False))
        malli = tee_malli().fit(X[opetusrivit], oikeat[opetusrivit])
        ennuste = malli.predict_proba(X[testi])
        for n, luokka in enumerate(malli.classes_):
            todnak[testi, PUOLUEET.index(luokka)] = ennuste[:, n]
    return oikeat, todnak


def aja_kausi(kausi, X_kaikki, fraaseja):
    """Kaikki toistot yhdelle kaudelle ilman tunnistettavimpia 10 %."""
    rng = np.random.default_rng(SIEMENET[kausi])
    rivit = ((indeksi["term"] == kausi).values & indeksi["party"].isin(PUOLUEET).values
             & (fraaseja >= MIN_FRAASEJA) & ~indeksi["person_id"].isin(POIS[(kausi, OSUUDET[0])]).values)
    puheet = indeksi[rivit].reset_index(drop=True).copy()
    puheet["label"] = puheet["party"].values
    tulos = []
    for toisto in range(TOISTOT + SATUNNAISET):
        sekoita = toisto >= TOISTOT
        oikeat, todnak = yksi_toisto(X_kaikki[rivit], puheet, rng, sekoita)
        tulos.append({"term": kausi, "series": "random" if sekoita else "real", "rep": toisto, **mittarit(oikeat, todnak)})
    return pd.DataFrame(tulos)


TIEDOSTO_B = TULOKSET + "tunnistettavat_uudelleenopetus.csv"
valmiit = pd.read_csv(TIEDOSTO_B) if os.path.exists(TIEDOSTO_B) and not UUDELLEEN else pd.DataFrame(columns=["term"])
puuttuvat = [k for k in KAUDET if k not in set(valmiit["term"])]
if puuttuvat:
    X_kaikki = sp.load_npz(KANSIO + "counts.npz").tocsr()
    fraaseja = np.asarray(X_kaikki.sum(axis=1)).ravel()
    for kausi in puuttuvat:
        valmiit = pd.concat([valmiit, aja_kausi(kausi, X_kaikki, fraaseja)], ignore_index=True)
        valmiit.to_csv(TIEDOSTO_B, index=False)
        print(kausi, "valmis")
osa_b = valmiit.copy()
print("Luettu:", TIEDOSTO_B, len(osa_b), "riviä")

Luettu: ../results/tunnistettavat_uudelleenopetus.csv 36 riviä


In [7]:
alkuperainen = pd.read_csv(TULOKSET + "party_summary.csv")[["term", "series", "rep", "accuracy", "polarisation"]]
vertailu = pd.concat([alkuperainen.assign(versio="kaikki edustajat (muistio 05)"),
                      osa_b.assign(versio="ilman tunnistettavinta 10%")])
for sarja, nimi in [("real", "oikeat puolueet"), ("random", "sekoitetut puolueet")]:
    print(f"Erottuvuus, {nimi}: keskiarvo (10.–90. persentiili)")
    display(yhteenveto(vertailu[vertailu["series"] == sarja], "polarisation"))

Erottuvuus, oikeat puolueet: keskiarvo (10.–90. persentiili)


term,2015-2019,2019-2023,2023-2027,muutos 2015–19 → 2023–27
versio,,,,
ilman tunnistettavinta 10%,0.271 (0.244–0.313),0.321 (0.238–0.384),0.390 (0.336–0.447),0.118
kaikki edustajat (muistio 05),0.391 (0.369–0.408),0.451 (0.402–0.485),0.558 (0.527–0.595),0.167


Erottuvuus, sekoitetut puolueet: keskiarvo (10.–90. persentiili)


term,2015-2019,2019-2023,2023-2027,muutos 2015–19 → 2023–27
versio,,,,
ilman tunnistettavinta 10%,-0.121 (-0.149–-0.099),-0.002 (-0.057–0.032),-0.104 (-0.197–0.002),0.017
kaikki edustajat (muistio 05),-0.003 (-0.079–0.099),0.001 (-0.049–0.065),-0.004 (-0.014–0.010),-0.000


In [8]:
print("Tarkkuus, oikeat puolueet: keskiarvo (10.–90. persentiili)")
yhteenveto(vertailu[vertailu["series"] == "real"], "accuracy")

Tarkkuus, oikeat puolueet: keskiarvo (10.–90. persentiili)


term,2015-2019,2019-2023,2023-2027,muutos 2015–19 → 2023–27
versio,,,,
ilman tunnistettavinta 10%,0.229 (0.217–0.240),0.279 (0.269–0.287),0.293 (0.282–0.303),0.064
kaikki edustajat (muistio 05),0.259 (0.253–0.268),0.301 (0.291–0.311),0.324 (0.308–0.336),0.064


## Tulkinta

**1. Tunnistettavimmat edustajat nostavat tasoa mutta eivät selitä kasvua.** Kun kunkin puolueen tunnistettavimmat 10 % poistetaan (21–23 edustajaa, 9–14 % kauden puheista), erottuvuus laskee kaikilla kausilla noin 0,04–0,05. Kausien välinen ero säilyy lähes ennallaan: kasvu kaudesta 2015–19 kauteen 2023–27 on 0,167 kaikilla edustajilla, 0,163 ilman tunnistettavinta 10 % ja 0,158 ilman tunnistettavinta 20 %. Tarkkuuden kasvu pysyy noin 6 prosenttiyksikössä kaikissa versioissa.

**2. Satunnainen poisto ei muuta tulosta.** Kun sama määrä edustajia poistetaan satunnaisesti, erottuvuus ja tarkkuus pysyvät samoina. Pudotus kohdassa 1 johtuu siis juuri tunnistettavimmista edustajista eikä pienemmästä edustajamäärästä.

**3. Uudelleenopetus (osa B) antaa saman kuvan.** Kun tunnistettavimmat 10 % poistetaan myös opetuksesta, taso laskee enemmän (0,27 / 0,32 / 0,39). Kasvu kaudesta 2015–19 kauteen 2023–27 on 0,12, suhteellisesti samaa luokkaa kuin alkuperäisessä (+44 % ja +43 %). Kausien 2015–19 ja 2023–27 vaihteluvälit eivät mene päällekkäin. Tarkkuuden kasvu on sama 6,4 prosenttiyksikköä kuin alkuperäisessä.

**4. Varaus: sekoitettujen nimikkeiden taso osassa B.** Sekoitetuilla nimikkeillä erottuvuus on kausilla 2015–19 ja 2023–27 noin −0,1, kun alkuperäisessä se on noin 0. Ristiinvalidoinnissa nollatilanteen AUC voi jäädä alle 0,5, kun luokissa on vähän ryhmiä, ja poisto vähentää edustajia puoluetta kohden. Neljä toistoa on lisäksi vähän: alkuperäisessäkin kauden 2015–19 arvot vaihtelivat välillä −0,09…0,15. Varaus ei muuta päätelmää, mutta osan B tasoja ei kannata verrata suoraan alkuperäisiin.

**Johtopäätös:** erottuvuuden kasvu ei johdu muutamasta helposti tunnistettavasta edustajasta. Se säilyy, kun tunnistettavimmat 10–20 % edustajista poistetaan, ja myös silloin, kun mallit opetetaan uudelleen ilman heitä.